# 🎀 봇에 물릴 vLLM 서버 띄우기 (도구 호출 + 터널)

> 브랜치 `feature/local-router` · 1단계 노트북(`vllm_qwen35.ipynb`)의 **후속**

## 1단계 노트북과 뭐가 다른가

| | 1단계 | 이 노트북 |
|---|---|---|
| 목적 | vLLM 서빙 자체를 경험 | **봇의 뇌로 실제 사용** |
| 도구 호출 | 없음 | **켬** (노션 도구 25개를 골라야 함) |
| 접근 범위 | Colab 안에서만 | **터널로 맥에서 접근** |
| 인증 | 없음 | **API 키 필수** |

목표는 하나다 — 맥에서 돌아가는 공주비서가 이 서버를 뇌로 쓰게 만드는 것.

## ⚠️ 보안: 이 노트북은 인터넷에 문을 연다

터널을 열면 `https://....trycloudflare.com` 주소가 생기고, **그 주소를 아는
사람은 누구나 아가씨 GPU로 추론을 돌릴 수 있다.** 그래서 vLLM을 `--api-key`와
함께 띄운다. 키 없는 요청은 401로 거절된다.

지켜야 할 것:
- 터널 주소와 API 키를 **아무 데도 붙여넣지 말 것** (깃허브·디스코드 포함)
- 실험이 끝나면 **런타임을 종료**해서 터널을 닫을 것
- `.env`는 gitignore 대상이니 거기 적는 건 안전하다

---
## 0. 설치 (런타임이 새것이면 필요)

1단계 노트북을 돌린 **같은 세션**이 살아있다면 이 절은 건너뛴다.
런타임이 새로 잡혔다면 처음부터 다시 깔아야 한다.

In [1]:
!pip install -q uv
!uv pip install --system vllm --torch-backend=auto \
    --extra-index-url https://wheels.vllm.ai/nightly

# torch/CUDA 버전 충돌 방지 (1단계에서 겪은 그 문제)
!pip uninstall -y torchaudio
!python -c "from vllm.engine.arg_utils import EngineArgs; print('✅ vLLM import OK')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 101.7 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 193 packages in 14.50s
Prepared 99 packages in 47.32s
Uninstalled 14 packages in 858ms
Installed 99 packages in 267ms
 + anthropic==0.120.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.3
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.2.1
 + depyf==0.20.0
 + detect-installer==0.1.0
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.139.0
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.22.2
 + fastar==0.11.0
 + fastsafetensors==0.3.3
 + flashinfer-python==0.6.15.post1
 + httpx-sse==0.4.3
 + humming-kernels==0.1.10
 + ijson==3.5.1
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - l

---
## 1. 도구 호출 파서 고르기

vLLM은 모델이 뱉은 텍스트에서 "도구를 부르겠다"는 부분을 **파서**로 뽑아내
OpenAI 형식(`tool_calls`)으로 바꿔준다. 모델 계열마다 표기법이 달라서
맞는 파서를 지정해야 한다.

Qwen3 문서는 `hermes`를 안내한다. Qwen3.5 전용 파서는 문서화돼 있지 않으므로,
**설치된 vLLM이 실제로 뭘 지원하는지 보고** 고른다.

In [2]:
# 이 vLLM 빌드가 아는 파서 목록. (버전에 따라 import 경로가 달라 두 방법을 시도)
names = None
for path in (
    "vllm.entrypoints.openai.tool_parsers",
    "vllm.entrypoints.serve.tool_parsers",
):
    try:
        mod = __import__(path, fromlist=["ToolParserManager"])
        names = sorted(mod.ToolParserManager.tool_parsers.keys())
        break
    except Exception:
        continue

if names:
    print("지원 파서:", names)
    print("\nqwen 관련:", [n for n in names if "qwen" in n.lower()])
else:
    print("목록을 못 읽었습니다. --help로 확인하세요:")
    !vllm serve --help 2>&1 | grep -i -A8 "tool-call-parser"

목록을 못 읽었습니다. --help로 확인하세요:


In [3]:
# Qwen3 계열 표준은 hermes. 위 목록에 qwen 전용 파서가 있으면 그쪽이 더 정확할 수 있다.
# 어느 쪽이 맞는지는 아래 '도구 호출 검증' 셀이 판정한다.
TOOL_PARSER = "hermes"
print("사용할 파서:", TOOL_PARSER)

사용할 파서: hermes


In [9]:
# 새로운 (XML 형식용)
TOOL_PARSER = "qwen3_xml"
print("사용할 파서:", TOOL_PARSER)

사용할 파서: qwen3_xml


---
## 2. 서버 띄우기 (도구 호출 + API 키)

1단계와 달라진 옵션 세 개:

| 옵션 | 왜 |
|---|---|
| `--enable-auto-tool-choice` | 모델이 스스로 도구를 고르게 허용 |
| `--tool-call-parser` | 뱉은 텍스트에서 도구 호출을 뽑아낼 파서 |
| `--api-key` | 터널로 열리므로 인증 필수 |

API 키는 여기서 무작위로 만든다. 출력에 찍히는 값을 나중에 `.env`에 넣는다.

### ⚠️ 먼저 포트를 비운다 (실제로 여기서 막혔다)

1단계 노트북의 서버가 8000번을 잡고 있으면 **새 서버는 바인딩에 실패해 즉시 죽는다.**
그런데 대기 셀은 옛 서버가 주는 200을 보고 "준비 완료"라 착각하고,
나중에 도구 요청을 보냈을 때야 이런 400이 터진다:

```
"auto" tool choice requires --enable-auto-tool-choice and --tool-call-parser to be set
```

플래그를 분명히 줬는데 이 에러가 나면 **다른 서버에 붙은 것**이다.

In [8]:
# 8000번을 잡고 있는 이전 vLLM을 정리한다.
import socket, time

!pkill -f "vllm serve"
time.sleep(5)

s = socket.socket()
free = s.connect_ex(("127.0.0.1", 8000)) != 0
s.close()
print("8000번 비었나:", free, "→ False면 아래 서버 셀을 돌려도 소용없다")

8000번 비었나: True → False면 아래 서버 셀을 돌려도 소용없다


In [10]:
import secrets, subprocess

MODEL   = "Qwen/Qwen3.5-4B"
PORT    = 8000
LOG     = "/content/vllm_bot.log"
API_KEY = "sk-" + secrets.token_urlsafe(24)   # 이 세션에서만 쓰는 임시 키

cmd = [
    "vllm", "serve", MODEL,
    "--port", str(PORT),
    "--tensor-parallel-size", "1",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--reasoning-parser", "qwen3",
    "--enable-auto-tool-choice",          # ← 도구 호출 허용
    "--tool-call-parser", TOOL_PARSER,    # ← 도구 호출 파서
    "--api-key", API_KEY,                 # ← 인증
]

logf = open(LOG, "w")
server = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
print("서버 시작. PID =", server.pid)
print("API_KEY =", API_KEY)
print("\n로딩에 5~8분 걸립니다. 다음 셀에서 기다립니다.")

서버 시작. PID = 8116
API_KEY = sk-***REDACTED***

로딩에 5~8분 걸립니다. 다음 셀에서 기다립니다.


In [11]:
import time, urllib.request, urllib.error

BASE  = f"http://localhost:{PORT}"
AUTH  = {"Authorization": f"Bearer {API_KEY}"}
ready = False

deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        print("\n❌ 서버가 죽었습니다. 로그 마지막 부분:\n")
        print(open(LOG).read()[-4000:])
        break
    try:
        req = urllib.request.Request(f"{BASE}/v1/models", headers=AUTH)
        with urllib.request.urlopen(req, timeout=3) as r:
            if r.status == 200:
                ready = True
                print("\n✅ 서버 준비 완료")
                break
    except Exception:
        pass
    print(".", end="")
    time.sleep(5)

if not ready:
    print("\n서버가 준비되지 않았습니다. 위 로그를 확인하세요.")
else:
    # 200을 받았다고 '이번에 띄운' 서버란 보장이 없다 — 옛 서버가 답했을 수 있다.
    # 새 서버만 --api-key를 갖고 있으니, 인증 없는 요청이 거절돼야 우리 서버다.
    try:
        urllib.request.urlopen(f"{BASE}/v1/models", timeout=5)
        print("⚠️ 인증 없이도 응답합니다 → 옛 서버에 붙었습니다.")
        print("   포트 정리 셀부터 다시 실행하세요.")
        ready = False
    except urllib.error.HTTPError as e:
        print(f"✅ 인증 검사 확인 (HTTP {e.code}) — 이번에 띄운 서버가 맞습니다")

...............................
✅ 서버 준비 완료
✅ 인증 검사 확인 (HTTP 401) — 이번에 띄운 서버가 맞습니다


---
## 3. 도구 호출 검증 — 여기가 관문

**이 셀이 통과해야 봇을 붙이는 의미가 있다.**

가짜 도구(`check_routine`) 하나를 정의해서 보내고, 모델이 그걸 부르겠다고
응답하는지 본다. 응답에 `tool_calls`가 담겨 오면 파서가 맞는 것이다.

실패하면 위 `TOOL_PARSER`를 다른 값(`qwen3_xml` 등)으로 바꾸고
**서버 셀부터** 다시 돌린다.

In [12]:
import json, urllib.error

TOOLS = [{
    "type": "function",
    "function": {
        "name": "check_routine",
        "description": "데일리루틴의 항목 하나를 완료 체크한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "item": {
                    "type": "string",
                    "description": "체크할 항목",
                    "enum": ["코테", "도착8시", "운동", "영어스피킹", "어드민나잇", "회고"],
                },
                "date": {"type": "string", "description": "YYYY-MM-DD"},
            },
            "required": ["item"],
        },
    },
}]

body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "오늘 운동 다녀왔어. 체크해줘."}],
    "tools": TOOLS,
    "tool_choice": "auto",
    "max_tokens": 300,
    "temperature": 0.7,
    "chat_template_kwargs": {"enable_thinking": False},
}
req = urllib.request.Request(
    f"{BASE}/v1/chat/completions",
    data=json.dumps(body).encode(),
    headers={**AUTH, "Content-Type": "application/json"},
)

# vLLM은 400을 낼 때 본문에 거부 사유를 담아준다. urllib은 예외를 던지며 그 본문을
# 버리므로, 반드시 직접 읽어서 보여준다. (사유를 못 보면 파서/옵션 중 뭐가 문제인지 알 수 없다)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
except urllib.error.HTTPError as e:
    print(f"❌ HTTP {e.code} — 서버가 요청을 거부했습니다. 사유:")
    print(e.read().decode()[:2000])
    out = None

if out:
    msg = out["choices"][0]["message"]
    if msg.get("tool_calls"):
        print("✅ 도구 호출 성공")
        for tc in msg["tool_calls"]:
            print("   ", tc["function"]["name"], tc["function"]["arguments"])
    else:
        print("❌ 도구를 안 불렀습니다. 파서를 바꿔서 서버를 다시 띄우세요.")
        print("   응답 내용:", msg.get("content"))

✅ 도구 호출 성공
    check_routine {"item": "운동"}


---
## 4. 터널 열기 — 맥에서 접근 가능하게

지금 서버는 Colab VM의 `localhost:8000`이라 맥에서 못 닿는다.
`cloudflared`가 공개 주소를 만들어 이 포트로 연결해준다.
(Gradio의 `share=True`가 쓰는 것과 같은 방식이다.)

In [13]:
!wget -q -O /content/cloudflared.deb \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared.deb > /dev/null 2>&1
!cloudflared --version

cloudflared version 2026.7.3 (built 2026-07-23-09:58 UTC)


In [14]:
import re, subprocess, time

TUN_LOG = "/content/cloudflared.log"
tunlog = open(TUN_LOG, "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=tunlog, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline:
    text = open(TUN_LOG).read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if m:
        PUBLIC_URL = m.group(0)
        break
    if tunnel.poll() is not None:
        print("❌ 터널이 죽었습니다:\n", text[-2000:])
        break
    print(".", end="")
    time.sleep(2)

print("\n공개 주소:", PUBLIC_URL)

...
공개 주소: https://***REDACTED***.trycloudflare.com


In [15]:
# 터널을 통해서도 실제로 닿는지 확인 (맥에서 볼 모습과 같은 경로)
import urllib.request, json

req = urllib.request.Request(f"{PUBLIC_URL}/v1/models", headers=AUTH)
with urllib.request.urlopen(req, timeout=30) as r:
    print("✅ 터널 경유 응답:", json.load(r)["data"][0]["id"])

✅ 터널 경유 응답: Qwen/Qwen3.5-4B


---
## 5. 맥에 붙이기

아래 출력 세 줄을 맥의 `.env` **맨 아래에 추가**한다.
(`.env`는 gitignore라 커밋되지 않는다.)

In [16]:
print(f"VLLM_BASE_URL={PUBLIC_URL}/v1")
print(f"VLLM_API_KEY={API_KEY}")
print(f"VLLM_MODEL={MODEL}")

VLLM_BASE_URL=https://***REDACTED***.trycloudflare.com/v1
VLLM_API_KEY=sk-***REDACTED***
VLLM_MODEL=Qwen/Qwen3.5-4B


붙여넣었으면 맥 터미널에서:

```bash
.venv/bin/python -m scripts.chat_cli
```

시작할 때 `뇌: 로컬 vLLM — Qwen/Qwen3.5-4B @ https://...` 가 찍히면 연결된 것이다.
Claude로 되돌리려면 `.env`의 `VLLM_BASE_URL` 줄만 지우면 된다.

### 무엇을 볼 것인가

터미널에 매 턴 이렇게 찍힌다:

```
🔧 도구 호출: attach_routine_photo({'item': '운동', ...})
↩️  결과[attach_routine_photo]: ...
```

관찰 포인트:
- **도구를 부르긴 하는가** (아예 안 부르고 잡담만 하는지)
- **맞는 도구를 고르는가** (노션 도구 25개 중에서)
- **인자를 제대로 채우는가** (날짜·항목 이름 — "어드민나잇"은 붙여 써야 한다)
- **응답 시간** (Claude 대비)

이게 라우터를 어디까지 맡길지 정하는 근거가 된다.

---
## 6. 정리

실험이 끝나면 **런타임을 종료**한다. 터널이 열린 채로 두면 안 된다.

Colab 메뉴 → 런타임 → 런타임 연결 해제 및 삭제

또는 아래 셀로 터널만 먼저 닫을 수 있다.

In [ ]:
# tunnel.terminate(); print("터널 닫음")
# server.terminate(); print("서버 종료")